# Check di `xi_KU_solver`

Questo notebook verifica passo per passo l'implementazione di `xi_KU_solver` in `Codes/main_SLURM.py` confrontandola con le eq. 41-55 di `Quantum_spin_squeezing_review.pdf`.

L'idea non e' solo controllare il valore finale di `xi_KU`, ma ricostruire tutti i conti intermedi:

- vettore di spin medio;
- base ortonormale `n0, n1, n2` costruita dal paper;
- matrice di covarianza dei momenti di spin;
- proiezione sul piano trasverso;
- `lambda_-` e angolo ottimo `phi_opt`;
- confronto finale tra formula del paper, implementazione del codice e minimizzazione brute-force.


In [3]:
from pathlib import Path
import importlib.util

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from qutip import ket2dm, tensor, sigmam

ROOT = Path.cwd()
MODULE_PATH = ROOT / 'main_SLURM.py'
if not MODULE_PATH.exists():
    MODULE_PATH = ROOT / 'Codes' / 'main_SLURM.py'

if not MODULE_PATH.exists():
    raise FileNotFoundError('Non trovo main_SLURM.py in cwd o in cwd/Codes')

spec = importlib.util.spec_from_file_location('main_slurm_module', MODULE_PATH)
ms = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ms)

print('Loaded module from:', MODULE_PATH)


Loaded module from: c:\Users\staka\OneDrive\Desktop\Figlio\Stochastic_Master_Thesis\Codes\main_SLURM.py


In [4]:
sigmam()

Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=CSR, isherm=False
Qobj data =
[[0. 0.]
 [1. 0.]]

## Mappa delle eq. 41-55

Nel notebook usiamo la seguente traduzione operativa delle formule del review:

- Eq. 41: `xi_KU^2 = (4/N) * min_{n_perp} (Delta J_{n_perp})^2`.
- Eq. 42-45: definizione della terna `n0, n1, n2` a partire dalla direzione dello spin medio.
- Eq. 46: `n_perp(phi) = n1 cos(phi) + n2 sin(phi)`.
- Eq. 47-52: costruzione della matrice di covarianza proiettata sul piano trasverso e diagonalizzazione.
- Eq. 53: `xi_KU^2 = (4/N) * lambda_-`, dove `lambda_-` e' il minimo autovalore della covarianza nel piano trasverso.
- Eq. 54-55: angolo ottimo `phi_opt = 0.5 * arctan2(B, A)` con `A = <J_n1^2> - <J_n2^2>` e `B = 2 Cov(J_n1, J_n2)`.

La tua funzione nel file Python implementa la stessa idea, ma lavora direttamente nella base cartesiana `Jx, Jy, Jz` e poi proietta la matrice di covarianza sul piano ortogonale al vettore medio.


In [2]:
def infer_N_from_state(rho):
    return int(round(np.log2(rho.shape[0])))


def covariance_matrix_xyz(rho):
    Jx_exp, Jy_exp, Jz_exp = ms.eval_spin_components(rho)
    Jmean = np.array(
        [
            float(np.real_if_close(Jx_exp)),
            float(np.real_if_close(Jy_exp)),
            float(np.real_if_close(Jz_exp)),
        ],
        dtype=float,
    )

    Js = [ms.J_x, ms.J_y, ms.J_z]
    C = np.zeros((3, 3), dtype=float)
    for i, Ji in enumerate(Js):
        for j, Jj in enumerate(Js):
            second_moment = 0.5 * ms.expect(Ji * Jj + Jj * Ji, rho)
            C[i, j] = float(np.real_if_close(second_moment - Jmean[i] * Jmean[j]))
    return Jmean, C


def basis_from_mean_spin(Jmean, tol=1e-12):
    m = np.linalg.norm(Jmean)
    if m <= tol:
        return None

    theta = np.arccos(np.clip(Jmean[2] / m, -1.0, 1.0))
    phi = np.arctan2(Jmean[1], Jmean[0])

    n0 = np.array([
        np.sin(theta) * np.cos(phi),
        np.sin(theta) * np.sin(phi),
        np.cos(theta),
    ])
    n1 = np.array([-np.sin(phi), np.cos(phi), 0.0])
    n2 = np.array([
        np.cos(theta) * np.cos(phi),
        np.cos(theta) * np.sin(phi),
        -np.sin(theta),
    ])

    return {
        'm': m,
        'theta': theta,
        'phi': phi,
        'n0': n0,
        'n1': n1,
        'n2': n2,
    }


def transverse_covariance(C_xyz, n1, n2):
    return np.array(
        [
            [float(n1 @ C_xyz @ n1), float(n1 @ C_xyz @ n2)],
            [float(n2 @ C_xyz @ n1), float(n2 @ C_xyz @ n2)],
        ],
        dtype=float,
    )


def variance_on_transverse_circle(C_xyz, n1, n2, phi_angle):
    n_perp = n1 * np.cos(phi_angle) + n2 * np.sin(phi_angle)
    return float(n_perp @ C_xyz @ n_perp)


def paper_xi_ku_breakdown(rho, tol=1e-12):
    N = infer_N_from_state(rho)
    Jmean, C_xyz = covariance_matrix_xyz(rho)
    basis = basis_from_mean_spin(Jmean, tol=tol)

    out = {
        'N': N,
        'Jmean': Jmean,
        'C_xyz': C_xyz,
        'xi_code': ms.xi_KU_solver(0, rho, N=N, tol=tol),
    }

    if basis is None:
        lam_min_full = float(np.linalg.eigvalsh(C_xyz)[0])
        out.update(
            {
                'mean_spin_zero': True,
                'lambda_min_full': lam_min_full,
                'xi_from_full_covariance': (4.0 / N) * lam_min_full,
            }
        )
        return out

    out.update(basis)

    C_perp = transverse_covariance(C_xyz, basis['n1'], basis['n2'])
    A = float(C_perp[0, 0] - C_perp[1, 1])
    B = float(2.0 * C_perp[0, 1])
    phi_opt = 0.5 * np.arctan2(B, A)
    lam_minus = 0.5 * ((C_perp[0, 0] + C_perp[1, 1]) - np.sqrt(A * A + B * B))

    out.update(
        {
            'mean_spin_zero': False,
            'C_perp': C_perp,
            'A': A,
            'B': B,
            'phi_opt': phi_opt,
            'lambda_minus': lam_minus,
            'xi_eq_41_55': (4.0 / N) * lam_minus,
        }
    )
    return out


def brute_force_xi_ku(rho, n_grid=4000):
    info = paper_xi_ku_breakdown(rho)
    if info['mean_spin_zero']:
        return info['xi_from_full_covariance'], None, None

    phi_grid = np.linspace(0.0, 2.0 * np.pi, n_grid, endpoint=False)
    vars_grid = np.array(
        [variance_on_transverse_circle(info['C_xyz'], info['n1'], info['n2'], x) for x in phi_grid]
    )
    idx = int(np.argmin(vars_grid))
    xi_brute = (4.0 / info['N']) * vars_grid[idx]
    return xi_brute, phi_grid[idx], vars_grid


## Stati di test

Uso tre stati campione:

- `CSS |++>`: caso di riferimento, deve dare `xi_KU = 1`.
- `CSS(theta=1.1, phi=0.3)`: altro stato coerente, ancora `xi_KU = 1`.
- `OAT from |++>, t=0.75`: stato non banale generato con `exp(-i t J_z^2)` per vedere un caso con squeezing effettivo.


In [3]:
oat_t = 0.75
U_oat = (-1j * oat_t * (ms.J_z ** 2)).expm()

test_states = {
    'CSS |++>': ket2dm(ms.PlusPlus),
    'CSS(theta=1.1, phi=0.3)': ket2dm(ms.css_2(1.1, 0.3)),
    f'OAT from |++>, t={oat_t}': ket2dm(U_oat * ms.PlusPlus),
}

selected_name = f'OAT from |++>, t={oat_t}'
rho = test_states[selected_name]

print('Selected state:', selected_name)


Selected state: OAT from |++>, t=0.75


In [4]:
info = paper_xi_ku_breakdown(rho)

summary = pd.Series(
    {
        'N': info['N'],
        '|<J>|': np.linalg.norm(info['Jmean']),
        'xi_code': info['xi_code'],
        'xi_eq_41_55': info.get('xi_eq_41_55', np.nan),
        'lambda_minus': info.get('lambda_minus', np.nan),
        'phi_opt': info.get('phi_opt', np.nan),
        'A': info.get('A', np.nan),
        'B': info.get('B', np.nan),
    }
)
display(summary.to_frame('value'))

display(pd.DataFrame(info['Jmean'].reshape(1, -1), columns=['Jx', 'Jy', 'Jz'], index=['<J>']))

if not info['mean_spin_zero']:
    basis_df = pd.DataFrame(
        np.vstack([info['n0'], info['n1'], info['n2']]),
        index=['n0', 'n1', 'n2'],
        columns=['x', 'y', 'z'],
    )
    display(basis_df)

display(pd.DataFrame(info['C_xyz'], index=['Jx', 'Jy', 'Jz'], columns=['Jx', 'Jy', 'Jz']))

if not info['mean_spin_zero']:
    display(pd.DataFrame(info['C_perp'], index=['n1', 'n2'], columns=['n1', 'n2']))


,value
N,2.000000e+00
|<J>|,7.316889e-01
xi_code,3.183612e-01
xi_eq_41_55,3.183612e-01
lambda_minus,1.591806e-01
phi_opt,-7.853982e-01
A,-5.551115e-17
B,-6.816388e-01


,Jx,Jy,Jz
<J>,0.731689,0.0,0.0


,x,y,z
n0,1.000000e+00,0.0,6.123234e-17
n1,-0.000000e+00,1.0,0.000000e+00
n2,6.123234e-17,0.0,-1.000000e+00


,Jx,Jy,Jz
Jx,0.464631,0.000000,0.000000
Jy,0.000000,0.500000,0.340819
Jz,0.000000,0.340819,0.500000


,n1,n2
n1,0.500000,-0.340819
n2,-0.340819,0.500000


## Confronto finale: codice vs paper vs brute force

Qui confrontiamo tre strade indipendenti:

- implementazione in `main_SLURM.py`;
- formula chiusa ricostruita dalle eq. 41-55;
- minimizzazione numerica diretta di `(Delta J_n_perp)^2` al variare di `phi`.


In [5]:
xi_brute, phi_brute, vars_grid = brute_force_xi_ku(rho)

comparison = pd.DataFrame(
    [
        {
            'xi_code': info['xi_code'],
            'xi_eq_41_55': info.get('xi_eq_41_55', np.nan),
            'xi_bruteforce': xi_brute,
            'abs(code-paper)': abs(info['xi_code'] - info.get('xi_eq_41_55', info['xi_code'])),
            'abs(code-brute)': abs(info['xi_code'] - xi_brute),
        }
    ]
)
display(comparison)

if not info['mean_spin_zero']:
    print('phi_opt from eq. 54 =', info['phi_opt'])
    print('phi_opt from brute force =', phi_brute)


,xi_code,xi_eq_41_55,xi_bruteforce,abs(code-paper),abs(code-brute)
0,0.318361,0.318361,0.318361,5.551115e-17,0.0


phi_opt from eq. 54 = -0.7853981633974484
phi_opt from brute force = 3.926990816987242


In [6]:
if not info['mean_spin_zero']:
    phi_grid = np.linspace(0.0, 2.0 * np.pi, len(vars_grid), endpoint=False)
    plt.figure(figsize=(8, 4.5))
    plt.plot(phi_grid, vars_grid, label=r'$(\Delta J_{n_\perp})^2$')
    plt.axvline(info['phi_opt'], color='tab:red', linestyle='--', label='phi_opt from eq. 54')
    plt.axhline(info['lambda_minus'], color='tab:green', linestyle=':', label='lambda_-')
    plt.xlabel(r'$\phi$')
    plt.ylabel(r'$(\Delta J_{n_\perp})^2$')
    plt.title(selected_name)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


C:\Users\campa\AppData\Local\Temp\ipykernel_1768\2175338599.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Check su piu' stati

Questo e' il controllo pratico piu' utile: facciamo girare lo stesso confronto su piu' stati e guardiamo l'errore residuo.


In [7]:
rows = []
for name, rho_test in test_states.items():
    info_test = paper_xi_ku_breakdown(rho_test)
    xi_brute_test, _, _ = brute_force_xi_ku(rho_test)
    rows.append(
        {
            'state': name,
            'xi_code': info_test['xi_code'],
            'xi_eq_41_55': info_test.get('xi_eq_41_55', np.nan),
            'xi_bruteforce': xi_brute_test,
            'abs(code-paper)': abs(info_test['xi_code'] - info_test.get('xi_eq_41_55', info_test['xi_code'])),
            'abs(code-brute)': abs(info_test['xi_code'] - xi_brute_test),
        }
    )

df_checks = pd.DataFrame(rows)
display(df_checks)


,state,xi_code,xi_eq_41_55,xi_bruteforce,abs(code-paper),abs(code-brute)
0,CSS |++>,1.000000,1.000000,1.000000,0.000000e+00,2.220446e-16
1,"CSS(theta=1.1, phi=0.3)",1.000000,1.000000,1.000000,3.330669e-16,6.661338e-16
2,"OAT from |++>, t=0.75",0.318361,0.318361,0.318361,5.551115e-17,0.000000e+00


## Caso speciale: `|<J>| = 0`

Le eq. 42-55 assumono che la direzione dello spin medio sia ben definita. Se `|<J>| = 0`, la terna `n0, n1, n2` non e' definita in modo univoco.

Nel tuo codice, in quel caso, si usa il minimo autovalore della matrice di covarianza completa `C`. Questo notebook controlla anche quel branch separatamente.


In [8]:
psi_singlet = (tensor(ms.gnd, ms.exc) - tensor(ms.exc, ms.gnd)).unit()
rho_singlet = ket2dm(psi_singlet)

info_singlet = paper_xi_ku_breakdown(rho_singlet)
display(pd.Series(info_singlet).to_frame('value'))


,value
N,2
Jmean,"[0.0, 0.0, 0.0]"
C_xyz,"[[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, ..."
xi_code,0.0
mean_spin_zero,True
lambda_min_full,0.0
xi_from_full_covariance,0.0


## Stress test su stati random

Per non limitarci a stati con struttura speciale, qui generiamo un campione riproducibile di stati casuali del sistema a due qubit, che e' il dominio naturale di `main_SLURM.py`.

Il confronto resta sempre lo stesso:

- `xi_code`: implementazione in `main_SLURM.py`;
- `xi_eq_41_55`: ricostruzione analitica delle eq. 41-55;
- `xi_bruteforce`: minimizzazione numerica diretta sul piano trasverso.

Uso sia stati puri random sia stati misti random, cosi' il check non dipende da proprieta' particolari del campione scelto.


In [9]:
from qutip import Qobj

subdims = list(ms.J_x.dims[0])
ket_dims = [subdims, [1] * len(subdims)]
dm_dims = [subdims, subdims]
hilbert_dim = int(np.prod(subdims))

def random_ket_two_qubits(rng):
    vec = rng.normal(size=hilbert_dim) + 1j * rng.normal(size=hilbert_dim)
    vec /= np.linalg.norm(vec)
    return Qobj(vec.reshape(hilbert_dim, 1), dims=ket_dims)

def random_dm_two_qubits(rng):
    G = rng.normal(size=(hilbert_dim, hilbert_dim)) + 1j * rng.normal(size=(hilbert_dim, hilbert_dim))
    rho = G @ G.conj().T
    rho /= np.trace(rho)
    return Qobj(rho, dims=dm_dims)

rng = np.random.default_rng(123456)
n_random_pure = 12
n_random_mixed = 12

random_states = {}
for idx in range(n_random_pure):
    random_states[f'random_pure_{idx:02d}'] = random_ket_two_qubits(rng)
for idx in range(n_random_mixed):
    random_states[f'random_mixed_{idx:02d}'] = random_dm_two_qubits(rng)

random_rows = []
for name, state in random_states.items():
    info_random = paper_xi_ku_breakdown(state)
    xi_brute_random, phi_brute_random, _ = brute_force_xi_ku(state, n_grid=8000)
    random_rows.append(
        {
            'state': name,
            'kind': 'pure' if state.isket else 'mixed',
            '|<J>|': float(np.linalg.norm(info_random['Jmean'])),
            'mean_spin_zero': bool(info_random['mean_spin_zero']),
            'xi_code': info_random['xi_code'],
            'xi_eq_41_55': info_random.get('xi_eq_41_55', info_random.get('xi_from_full_covariance', np.nan)),
            'xi_bruteforce': xi_brute_random,
            'phi_opt': info_random.get('phi_opt', np.nan),
            'phi_bruteforce': phi_brute_random if phi_brute_random is not None else np.nan,
            'abs(code-paper)': abs(info_random['xi_code'] - info_random.get('xi_eq_41_55', info_random.get('xi_from_full_covariance', info_random['xi_code']))),
            'abs(code-brute)': abs(info_random['xi_code'] - xi_brute_random),
            'abs(paper-brute)': abs(info_random.get('xi_eq_41_55', info_random.get('xi_from_full_covariance', info_random['xi_code'])) - xi_brute_random),
        }
    )

df_random_checks = pd.DataFrame(random_rows).sort_values(['kind', 'state']).reset_index(drop=True)
display(df_random_checks)


,state,kind,|<J>|,mean_spin_zero,xi_code,xi_eq_41_55,xi_bruteforce,phi_opt,phi_bruteforce,abs(code-paper),abs(code-brute),abs(paper-brute)
0,random_mixed_00,mixed,0.196213,False,0.666716,0.666716,0.666716,0.796823,2.367975,3.330669e-16,2.684570e-08,2.684570e-08
1,random_mixed_01,mixed,0.229099,False,1.111110,1.111110,1.111110,-1.075496,0.495586,2.220446e-16,2.021447e-08,2.021447e-08
2,random_mixed_02,mixed,0.181239,False,0.513875,0.513875,0.513875,0.809968,2.380542,0.000000e+00,3.696185e-08,3.696185e-08
3,random_mixed_03,mixed,0.301291,False,0.800505,0.800505,0.800505,0.043730,1.614779,3.330669e-16,1.145081e-08,1.145081e-08
4,random_mixed_04,mixed,0.141401,False,0.816459,0.816459,0.816459,-0.244013,1.326537,2.220446e-16,1.611001e-08,1.611001e-08
5,random_mixed_05,mixed,0.337588,False,0.661027,0.661027,0.661027,0.251531,1.822124,0.000000e+00,1.665780e-08,1.665780e-08
6,random_mixed_06,mixed,0.372218,False,0.625039,0.625039,0.625039,0.428534,5.141216,0.000000e+00,1.643616e-08,1.643616e-08
7,random_mixed_07,mixed,0.256938,False,0.909189,0.909189,0.909189,1.030081,5.742831,1.110223e-16,2.344506e-08,2.344506e-08
8,random_mixed_08,mixed,0.284606,False,0.721963,0.721963,0.721963,-0.981780,0.589049,3.330669e-16,3.681530e-10,3.681533e-10
9,random_mixed_09,mixed,0.143457,False,0.656628,0.656628,0.656628,-1.406989,0.164148,0.000000e+00,8.823850e-08,8.823850e-08


In [ ]:
random_summary = pd.Series(
    {
        'n_states': len(df_random_checks),
        'n_pure': int((df_random_checks['kind'] == 'pure').sum()),
        'n_mixed': int((df_random_checks['kind'] == 'mixed').sum()),
        'max |code-paper|': df_random_checks['abs(code-paper)'].max(),
        'max |code-brute|': df_random_checks['abs(code-brute)'].max(),
        'max |paper-brute|': df_random_checks['abs(paper-brute)'].max(),
        'mean |code-paper|': df_random_checks['abs(code-paper)'].mean(),
        'mean |code-brute|': df_random_checks['abs(code-brute)'].mean(),
        'mean |paper-brute|': df_random_checks['abs(paper-brute)'].mean(),
    }
)
display(random_summary.to_frame('value'))

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(df_random_checks))
eps = 1e-18
ax.semilogy(x, df_random_checks['abs(code-paper)'] + eps, 'o-', label='|code-paper|')
ax.semilogy(x, df_random_checks['abs(code-brute)'] + eps, 's-', label='|code-brute|')
ax.semilogy(x, df_random_checks['abs(paper-brute)'] + eps, '^-', label='|paper-brute|')
ax.set_xlabel('random state index')
ax.set_ylabel('absolute error')
ax.set_title('Confronto errori sui random states')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()


## Lettura pratica del risultato

Se i due errori `abs(code-paper)` e `abs(code-brute)` sono numericamente piccoli (tipicamente ordine `1e-12` o piu' piccoli), allora la funzione `xi_KU_solver` e' coerente con la costruzione delle eq. 41-55 per gli stati con spin medio non nullo.

Per gli stati con `|<J>| = 0`, il notebook separa esplicitamente il branch usato dal codice, che e' una estensione naturale ma non e' la parametrizzazione diretta delle eq. 42-55.
